# Mestres do XGH — Aula 12
## APIs e `requests`: buscar dado que não é seu

**Shift + Enter** roda a célula.

Sábado passado você instalou biblioteca que lê arquivo do **seu** computador.
Hoje você vai ler dado do computador **de outra pessoa**.

> **A frase da aula:** status 200 não quer dizer que deu certo.

---
# 0. Criando os arquivos da aula

Rode esta célula primeiro. Ela gera a planilha `clientes.csv`, com uma coluna de CEP
que a gente vai completar no fim da aula.

Repare que tem CEP bom, CEP que não existe e CEP malformado. **De propósito.**

In [ ]:
import csv, random

random.seed(12)

nomes = ["Ana Costa", "Bruno Sales", "Carla Dias", "Diego Matos",
         "Elaine Reis", "Fábio Luz", "Gisele Prado", "Heitor Nunes"]

ceps = ["11013001",   # Santos — existe
        "11440003",   # Guarujá — existe
        "11701000",   # Praia Grande — existe
        "11310000",   # São Vicente — existe
        "11075350",   # Santos — existe
        "01311000",   # São Paulo — existe
        "99999999",   # formato certo, mas NÃO existe
        "123"]        # malformado

random.shuffle(nomes)

with open("clientes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["nome", "cep"])
    for nome, cep in zip(nomes, ceps):
        w.writerow([nome, cep])

print("clientes.csv criado")
print(open("clientes.csv", encoding="utf-8").read())

---
# 1. A pergunta que ficou de sábado passado

*"Instalei uma biblioteca que busca dado da internet — de onde vem esse dado?
Quem está do outro lado? E por que de graça?"*

### Abra estes dois links em abas separadas

1. O site dos Correios, buscando um CEP — [buscacepinter.correios.com.br](https://buscacepinter.correios.com.br/app/endereco/index.php)
2. Isto: [viacep.com.br/ws/11013001/json/](https://viacep.com.br/ws/11013001/json/)

**Os dois têm o mesmo dado.** Um está bonito, com menu, logo e botão. O outro está feio e é isso:

```json
{
  "cep": "11013-001",
  "logradouro": "Avenida João Pessoa",
  "bairro": "Centro",
  "localidade": "Santos",
  "uf": "SP",
  "ddd": "13"
}
```

Você prefere o primeiro porque você é **gente**. Seu programa não é gente.
Para ele, o primeiro é um pesadelo e o segundo é um presente embrulhado.

### Isso é uma API

Um endereço na internet que devolve **dado puro**, no formato que programa lê.
Sem menu, sem cor, sem botão. O dado e mais nada.

> **Site é para o olho. API é para o `for`.**

---
# 2. A primeira chamada

Sábado passado, a tabela de nome-de-instalação × nome-de-import.
Hoje, a exceção boazinha: **os dois batem.**

```
pip install requests
```

No Colab, ou direto numa célula:

In [ ]:
%pip install requests --quiet

Agora a aula inteira em três linhas:

In [ ]:
import requests

resposta = requests.get("https://api.agify.io?name=guilherme")
print(resposta)

```
<Response [200]>
```

### O que aconteceu na rede, em quatro passos

1. o Python abriu uma conexão com o servidor `api.agify.io`
2. mandou um pedido: *"me dá o que está em `/?name=guilherme`"*
3. o servidor respondeu com um **número de status** e um **corpo de texto**
4. o `requests` embrulhou as duas coisas num objeto e devolveu

**Esse objeto não é o dado. É o envelope.** Dentro dele tem o status e tem o corpo.

### O corpo cru

In [ ]:
print(resposta.text)

```
{"count":5405,"name":"guilherme","age":34}
```

Chaves, dois-pontos, vírgula: **isso é JSON** — a Aula 7. Você já sabe ler isso.

A API olhou 5405 pessoas chamadas Guilherme e apostou que eu tenho 34 anos. Errou.
Mas repare no `count`: ela te disse **em quantos dados se baseou**. Isso é uma API honesta.

### Agora com o seu nome

In [ ]:
meu_nome = "guilherme"   # troque pelo seu

r = requests.get("https://api.agify.io?name=" + meu_nome)
print(r.json())

---
# 3. `status_code` e `.json()`

## O status: o que o servidor achou do seu pedido

| Status | Quer dizer | Em português de gente |
|---|---|---|
| **200** | OK | o servidor entendeu e respondeu |
| **400** | Bad Request | você pediu errado |
| **401 / 403** | Unauthorized / Forbidden | você não provou quem é, ou não pode |
| **404** | Not Found | esse endereço não existe |
| **429** | Too Many Requests | você pediu demais, espere |
| **500** | Internal Server Error | **o problema é do outro lado**, não seu |

**Decore três:** `200` deu certo, `4xx` o erro é seu, `5xx` o erro é dele.
Isso resolve 95% dos casos.

In [ ]:
print(resposta.status_code)

## O `.json()`: o reencontro com a Aula 7

In [ ]:
dados = resposta.json()

print(type(dados))
print(dados["age"])
print(dados["count"])

```
<class 'dict'>
34
5405
```

Olhe o tipo. **`dict`.** É dicionário — a mesma coisa da Aula 7, com as mesmas chaves e os
mesmos colchetes.

**Você não precisa aprender nada novo para usar API.** Você já sabe navegar nisso desde setembro.

### `.text` ou `.json()`? Não confunda

| | o que é | o que você faz com ele |
|---|---|---|
| `resposta.text` | **string** | `print`, procurar pedaço, salvar em arquivo |
| `resposta.json()` | **dicionário ou lista** | `dados["chave"]`, `for`, `.get()` |

`.text` é o papel. `.json()` é o papel já lido e organizado.
É o mesmo pulo do `json.loads` da Aula 7 — só que o `requests` já faz para você.

---
# 4. Parâmetros na URL

```
https://api.agify.io?name=guilherme
                    ↑         ↑
                    │         └── o valor
                    └── começa a lista de parâmetros
```

Aquele `?` separa o endereço dos **parâmetros**. Depois dele vem `chave=valor`,
e `&` para juntar mais de um. É assim que **uma** URL atende mil pedidos diferentes.

### Dois jeitos de montar — e por que o segundo é melhor

In [ ]:
# jeito 1: colando na mão (funciona, mas quebra fácil)
nome = "maria"
r1 = requests.get("https://api.agify.io?name=" + nome)
print(r1.url)

# jeito 2: deixando o requests montar
r2 = requests.get("https://api.agify.io",
                  params={"name": "maria", "country_id": "BR"})
print(r2.url)
print(r2.json())

```
https://api.agify.io/?name=maria
https://api.agify.io/?name=maria&country_id=BR
{'count': 3622, 'name': 'maria', 'age': 59, 'country_id': 'BR'}
```

O `params=` monta a URL por você. **Por que isso importa?** Porque no dia em que o valor
tiver um espaço, um acento ou um `&`, o jeito 1 monta uma URL quebrada e você perde uma hora
procurando o erro no lugar errado. O `requests` sabe escapar; você não precisa saber.

E repare: mesma API, mesmo endereço, **parâmetro diferente, resposta diferente**.
Maria no Brasil tem 59.

> Ler a documentação de uma API é, quase sempre, descobrir **que parâmetros ela aceita**.

### Nem toda API usa `?`

```
https://viacep.com.br/ws/11013001/json/
                         ↑         ↑
                         CEP       formato
```

O ViaCEP coloca o CEP **no meio do endereço**. Não tem regra universal — tem documentação.
De novo: é a habilidade de sábado passado.

---
# 5. 💥 Os quatro jeitos de quebrar

`requests.get` funciona na primeira tentativa. É o resto que vai te morder.

**O nº 1 é o mais importante desta aula** — e o mais perigoso do curso até aqui,
porque ele não parece erro.

## 💥 nº 1 — o erro que não é erro

Primeiro o caso que funciona, para ter referência:

In [ ]:
r = requests.get("https://viacep.com.br/ws/11013001/json/", timeout=5)

print(r.status_code)
print(r.json())

### Agora pare e aposte, antes de rodar a próxima célula

Vou pedir um CEP que **não existe**: `99999999`.
Oito dígitos, formato certo, mas não existe endereço nenhum com esse CEP.

**Que status você aposta que vem?** Escreva num papel. Sério.

*(a maioria da turma aposta 404)*

In [ ]:
r = requests.get("https://viacep.com.br/ws/99999999/json/", timeout=5)

print(r.status_code)
print(r.json())

```
200
{'erro': 'true'}
```

## Duzentos.

O servidor te disse que **deu tudo certo**. E dentro do envelope não tem endereço nenhum.

Se o seu programa fosse `if r.status_code == 200: usa o dado`, ele teria usado.
E teria escrito no relatório, ou no banco, ou no e-mail do cliente, **um endereço vazio**.
Sem erro. Sem aviso. Sem stack trace.

### Lembra da Aula 10?

É o mesmo monstro de duas aulas atrás, de roupa nova.

- **Aula 10:** o pandas calculou média ignorando os vazios e te deu um **número errado com cara
  de número certo**
- **Hoje:** a API te deu **sucesso com cara de sucesso e dado nenhum dentro**

Os erros que gritam são os baratos — você lê a mensagem e conserta.
Os caros são estes: os que passam.

---

> ## Status 200 não quer dizer que deu certo.
> ### Status 200 quer dizer que o servidor respondeu.
> Quem decide se deu certo é **você**, olhando o conteúdo.

---

### O conserto são duas linhas

In [ ]:
dados = r.json()

if r.status_code != 200:
    print("o servidor recusou:", r.status_code)
elif "erro" in dados:
    print("CEP não encontrado")
else:
    print(dados["logradouro"], "-", dados["localidade"])

```
CEP não encontrado
```

Duas checagens: primeiro o status, **depois o conteúdo**.

E `"erro" in dados` é o `in` de dicionário da Aula 7 — você já tinha a ferramenta,
faltava saber que precisava dela.

### E por que o ViaCEP faz isso?

Porque **para o servidor nada deu errado.** Ele recebeu um pedido válido, consultou a base,
não achou, e respondeu. Do ponto de vista dele a entrega foi perfeita.

Cada API escolhe o seu jeito, e você descobre **lendo a documentação ou testando** —
de preferência os dois.

## 💥 nº 2 — `.json()` explodindo

Agora um CEP **malformado**: `123`. Três dígitos. Nem parece CEP.

A célula abaixo **vai dar erro**. Leia a mensagem:

In [ ]:
r = requests.get("https://viacep.com.br/ws/123/json/", timeout=5)

print(r.status_code)
print(r.json())

```
400
requests.exceptions.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
```

Dois fatos novos. O status virou **400** — *você pediu errado*, que é honesto.
E o `.json()` **explodiu**. Por quê?

In [ ]:
print(r.text[:90])

Veio **HTML**. Uma página de erro, feita para pessoa ver.
O `.json()` tentou ler `<` como se fosse JSON e desistiu na primeira letra —
é literalmente o que a mensagem diz: *expecting value, line 1, column 1*.

> **`JSONDecodeError` quase nunca é erro do seu código.** É o servidor te mandando outra coisa:
> página de erro, página de login, aviso de manutenção.

### O `try/except` da Aula 9 virando obrigação

Lá na Aula 9 o `try/except` era zelo — coisa de programador cuidadoso.
Em código que fala com a internet **virou obrigação**, porque agora o seu programa depende de
uma máquina que você não controla, numa rede que você não controla.

In [ ]:
try:
    dados = r.json()
except requests.exceptions.JSONDecodeError:
    print("o servidor não mandou JSON. Status:", r.status_code)
    dados = None

## 💥 nº 3 — 429, e o limite que não é seu bug

In [ ]:
r = requests.get("https://economia.awesomeapi.com.br/json/last/USD-BRL", timeout=5)

print(r.status_code)
print(r.text)

Rodando isto na preparação da aula, veio:

```
429
{"status":429,"code":"QuotaExceeded","message":"Quota exceeded. To continue using the service, please see https://docs.awesomeapi.com.br/aviso-sobre-limites"}
```

**429: você pediu demais.** Não tem nada de errado no código. A API de cotação é de graça até
um certo número de chamadas por hora, e alguém já gastou.

Três coisas:

1. **Na sua máquina pode vir `200` com a cotação**, porque o limite é contado por quem pede e
   quando. Se vier 200, ótimo. Se vier 429, você acabou de ver ao vivo
2. Quando vier 429, a resposta certa é **esperar**, não insistir. Programa que vê 429 e tenta de
   novo na hora só piora
3. E a mais importante: **não existe API de graça.** Existe API que alguém paga e deixa você usar
   um pouquinho. O 429 é o momento em que esse pouquinho acabou

## 💥 nº 4 — timeout, o travamento silencioso

Esse é diferente dos três. Nos outros o programa **erra**. Nesse ele **não faz nada**, para sempre.

Imagine: o servidor aceita a conexão e nunca responde. Não é erro, não é 500 — é silêncio.

**Quanto tempo o `requests.get` espera?** Sem `timeout`: **para sempre.**
Seu programa fica pendurado e você não sabe se travou ou se está trabalhando.

A célula abaixo pede mil vezes mais rápido do que a internet consegue, só para você ver o erro:

In [ ]:
r = requests.get("https://viacep.com.br/ws/11013001/json/", timeout=0.001)

```
requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='viacep.com.br', port=443):
Read timed out. (read timeout=0.001)
```

Mas repare: **ele deu erro.** É isso que a gente quer. Erro é tratável; travamento não é.

---

> ## Todo `requests.get` da sua vida leva `timeout=`.
> Cinco ou dez segundos. Sempre.

Semana que vem a gente vai fazer script rodar sozinho de madrugada.
Um `get` sem `timeout` lá dentro é um programa que amanhece pendurado.

---
# 6. Chave de API: por que existe

O 429 abriu a porta. Se a API é de graça, **como é que eles sabem que fui eu que pedi demais?**

Pelo endereço de internet, por enquanto. Que funciona mal: a sala toda sai pelo mesmo endereço,
e um aluno pedindo demais derruba os outros cinco. Para APIs maiores isso não serve.
Então eles pedem que você **se identifique**.

> **Chave de API é uma senha que identifica o seu programa**, não você.
> É um texto comprido, feio, que você coloca na requisição.

```python
# formato típico — não rode, é só para ver a cara
requests.get("https://api.exemplo.com/dados",
             headers={"Authorization": "Bearer sk-9f2c...a81"},
             timeout=10)
```

### Por que existe

| Para quem mantém a API | Na prática |
|---|---|
| **Contar** | saber quantas chamadas são suas, para aplicar o limite |
| **Cobrar** | tier gratuito até X, pago depois |
| **Bloquear** | se alguém abusar, desligam **aquela** chave, não a API toda |
| **Saber quem usa** | o que mais chamam, o que podem desativar |

### A regra que vale o resto do curso

A chave é **sua**, e ela identifica você. Se alguém pegar a sua, gasta o seu limite e,
se for paga, **gasta o seu dinheiro**.

- chave **não vai** dentro do código
- chave **não vai** para o GitHub
- chave **não vai** no print de tela que você manda no grupo

Na **Aula 14** a gente aprende git, e você vai ver que **o git lembra para sempre** — apagar a
chave num commit seguinte não tira do histórico. Na **Aula 18** você vai ter uma chave de LLM de
verdade na mão.

Hoje nada do que a gente usou precisa de chave. ViaCEP, Agify, cotação — todos abertos.
É de propósito: hoje era para aprender a conversar. A chave aparece quando a conversa fica cara.

---
# 7. 🛠️ Mão na massa: buscador de CEP

Um programa que pergunta o CEP com `input()` e imprime o endereço formatado.
E que **não quebra** em nenhum dos três casos de hoje:

1. CEP certo → imprime rua, bairro, cidade, UF
2. CEP que não existe → `200` com `{"erro": "true"}` → diz *"CEP não encontrado"*
3. CEP malformado, servidor fora, internet caída → trata e avisa, **sem stack trace na cara do usuário**

### Os três CEPs de teste

```
11013001   →  deve achar (Santos)
99999999   →  não existe
123        →  malformado
```

**Enquanto não passar nos três, não está pronto.** É o mesmo critério de código profissional:
o caminho felizinho é a parte fácil.

---

**Tente antes de rolar para baixo.** A resposta está depois.

In [ ]:
# Seu espaço


<details>
<summary>Clique só depois de tentar</summary>

```python
import requests

def buscar_cep(cep):
    cep = cep.strip().replace("-", "").replace(".", "")
    url = f"https://viacep.com.br/ws/{cep}/json/"
    try:
        r = requests.get(url, timeout=5)
    except requests.exceptions.RequestException as e:
        return None, f"não deu para falar com o servidor ({type(e).__name__})"

    if r.status_code != 200:
        return None, f"o servidor recusou o pedido (status {r.status_code})"

    try:
        dados = r.json()
    except requests.exceptions.JSONDecodeError:
        return None, "o servidor não mandou JSON"

    if "erro" in dados:
        return None, "CEP não encontrado"

    return dados, None
```

**Repare numa escolha de projeto:** a função devolve **duas coisas** — o dado e o problema.
Um dos dois é sempre `None`. E a função **não dá `print` de nada**.

Por que isso é melhor? Porque amanhã essa função vai rodar dentro de um laço com mil CEPs, e
ninguém quer mil `print` no meio do relatório.

> **Função busca; quem chamou decide.**
</details>

### A função (rode esta célula para ter a `buscar_cep` disponível)

In [ ]:
import requests

def buscar_cep(cep):
    cep = cep.strip().replace("-", "").replace(".", "")
    url = f"https://viacep.com.br/ws/{cep}/json/"

    try:
        r = requests.get(url, timeout=5)
    except requests.exceptions.RequestException as e:
        return None, f"não deu para falar com o servidor ({type(e).__name__})"

    if r.status_code != 200:
        return None, f"o servidor recusou o pedido (status {r.status_code})"

    try:
        dados = r.json()
    except requests.exceptions.JSONDecodeError:
        return None, "o servidor não mandou JSON"

    if "erro" in dados:
        return None, "CEP não encontrado"

    return dados, None


def mostrar(cep):
    dados, problema = buscar_cep(cep)
    if problema:
        print("❌", problema)
    else:
        print(f"{dados['logradouro']}, {dados['bairro']}")
        print(f"{dados['localidade']} - {dados['uf']}  (DDD {dados['ddd']})")


for teste in ["11013001", "99999999", "123"]:
    print(f"--- {teste} ---")
    mostrar(teste)

```
--- 11013001 ---
Avenida João Pessoa, Centro
Santos - SP  (DDD 13)
--- 99999999 ---
❌ CEP não encontrado
--- 123 ---
❌ o servidor recusou o pedido (status 400)
```

### Agora a versão que pergunta

*(no Colab, a caixa de texto aparece embaixo da célula)*

In [ ]:
cep = input("Digite o CEP: ")
mostrar(cep)

---
# 8. 🎁 Bônus: completando a planilha *(o pandas da Aula 10 volta)*

A `clientes.csv` tem nome e CEP. Faltam as colunas de endereço.

Oito linhas, oito chamadas. **Duas vão falhar** — e você sabe exatamente quais e por quê.

In [ ]:
import pandas as pd

df = pd.read_csv("clientes.csv", dtype={"cep": str})

logradouros, cidades, ufs, situacoes = [], [], [], []

for cep in df["cep"]:
    dados, problema = buscar_cep(cep)
    if problema:
        logradouros.append(None)
        cidades.append(None)
        ufs.append(None)
        situacoes.append(problema)
    else:
        logradouros.append(dados["logradouro"])
        cidades.append(dados["localidade"])
        ufs.append(dados["uf"])
        situacoes.append("ok")

df["logradouro"] = logradouros
df["cidade"] = cidades
df["uf"] = ufs
df["situacao"] = situacoes

print(df.to_string())

### E agora o reflexo da Aula 10

Antes de confiar na tabela, **conte os buracos**:

In [ ]:
print(df.isnull().sum())
print()
print(df["situacao"].value_counts())

O relatório ficou com buraco — e você sabe exatamente **qual** buraco e **por quê**.

É isso que um programa bom faz: ele não esconde a falha, ele a **localiza**.

---
# 9. O que isso destrava

Com `requests.get` e `.json()`, estas coisas saíram de *impossível* e viraram *uma tarde*:

- **CEP, endereço, cotação, clima, feriado** — dado público, de graça
- **Qualquer sistema do seu trabalho** que tenha API: puxar dado sem abrir o site, sem exportar planilha
- **Juntar duas fontes** que ninguém juntou ainda — é aí que mora quase todo projeto interessante
- **Chamar IA de dentro do seu código** — a Aula 18 é exatamente isto, com outra URL e uma chave

E repare no tamanho do que você aprendeu hoje: `requests.get`, `status_code`, `.json()`,
`params`, `timeout`.

**Cinco coisas.** A API do ChatGPT se usa com essas mesmas cinco.

---
# 10. 📝 Tarefa da semana

**1.** Terminar o buscador de CEP e provar que passa nos três casos — `11013001`, `99999999`,
`123`. Mande no grupo o print das três saídas.

**2.** Trocar o ViaCEP por **outra API aberta** e fazer funcionar. Sugestões:

- `api.agify.io` — idade por nome
- `brasilapi.com.br` — feriado, banco, DDD, CNPJ
- `open-meteo.com` — clima

**Leia a documentação sozinho.** É a habilidade da Aula 11 outra vez.

**3.** Achar uma API que resolva **um problema seu de verdade**. Não precisa programar: mande no
grupo qual você achou, o link da documentação e **o que você faria com ela**.

**4. Desafio:** pegar os CEPs da `clientes.csv`, completar o endereço de cada um e salvar um
`clientes_completo.csv` com a coluna `situacao` dizendo o que aconteceu em cada linha
(`ok`, `não encontrado`, `erro`). É o pandas da Aula 10 encontrando a API de hoje.

---

## A pergunta que fica

Olhe o que você tem agora: um programa que busca dado de fora, trata os erros e monta um
relatório. **Isso é software de verdade.**

Mas ele tem um defeito que você não vai mais conseguir ignorar:

### ele só roda quando você aperta o play.

Você precisa abrir o computador, abrir o VS Code e clicar.

E se a cotação precisa ser consultada todo dia às 9h?
E se o relatório precisa sair na segunda de manhã, **sem você**?

**Como é que isso roda sozinho?**

---

*Mestres do XGH · Aula 12 · 31/10/2026*

In [ ]:
# Tarefa 2 — outra API, lendo a documentação sozinho
